Some hyper parameters have been reduced due to computer limitiations, as I do  not have access to strong gpus only a good cpu.

In [1]:
import os, random, time, itertools, json, warnings
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib
matplotlib.use('Agg')          
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats        
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import datasets, transforms, models
from sklearn.cluster import MiniBatchKMeans, KMeans
from sklearn.neighbors import NearestNeighbors
from sklearn.manifold import TSNE

print('Imports OK')
print('torch', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

Imports OK
torch 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


In [2]:
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

DEVICE   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SAVE_DIR = '/kaggle/working/typiclust_ckpts'
PLOT_DIR = '/kaggle/working/plots'
os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(PLOT_DIR, exist_ok=True)

CFG = {
    # SimCLR self-supervised pre-training
    'simclr_epochs'      : 100, 
    'simclr_lr'          : 0.4,
    'simclr_batch'       : 512,
    'simclr_temperature' : 0.5,
    'simclr_proj_dim'    : 128,
    'simclr_momentum'    : 0.9,
    'simclr_wd'          : 1e-4,

    # Framework A – Fully Supervised
    'sup_epochs'         : 70,
    'sup_lr'             : 0.025,
    'sup_batch'          : 128,
    'sup_momentum'       : 0.9,

    # Framework B – Linear Probe on SimCLR embeddings
    'lp_epochs'          : 50,    # paper: 100  [↓2×]
    'lp_lr'              : 2.5,
    'lp_batch'           : 256,

    # Framework C – Semi-Supervised (FixMatch-lite) 
    'ssl_iters'          : 30_000, 
    'ssl_lr'             : 0.03,
    'ssl_batch_l'        : 64,
    'ssl_batch_u'        : 64,
    'ssl_threshold'      : 0.95,
    'ssl_wd'             : 5e-4,

    # Active-Learning settings
    'al_budget'          : 10, 
    'al_iterations'      : 5,  
    'typi_k'             : 20,   
    'max_clusters'       : 500,  
}

print('Device:', DEVICE)
print('Save dir:', SAVE_DIR)
print('Config loaded. Total label budget:', CFG['al_budget'] * CFG['al_iterations'])

Device: cuda
Save dir: /kaggle/working/typiclust_ckpts
Config loaded. Total label budget: 50


In [3]:
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2470, 0.2435, 0.2616)

tf_plain = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])
tf_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])
tf_weak = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])
tf_strong = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.4, 0.4, 0.4, 0.1),
    transforms.RandomGrayscale(p=0.2),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])
# SimCLR augmentation: random resized crop + colour jitter + grayscale
tf_simclr = transforms.Compose([
    transforms.RandomResizedCrop(32, scale=(0.2, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.4, 0.4, 0.4, 0.1),
    transforms.RandomGrayscale(p=0.2),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])

DATA_ROOT = '/kaggle/working/data'
os.makedirs(DATA_ROOT, exist_ok=True)

raw_train = datasets.CIFAR10(DATA_ROOT, train=True,  download=True)
raw_test  = datasets.CIFAR10(DATA_ROOT, train=False, download=True)

N_TRAIN       = len(raw_train)   # 50 000
N_TEST        = len(raw_test)    # 10 000
N_CLS         = 10
ALL_TRAIN_IDX = list(range(N_TRAIN))
CLASSES       = raw_train.classes 

class PlainDataset(Dataset):
    def __init__(self, raw, transform):
        self.raw = raw
        self.tf  = transform
    def __len__(self):
        return len(self.raw)
    def __getitem__(self, i):
        img, label = self.raw[i]
        return self.tf(img), label

test_loader = DataLoader(
    PlainDataset(raw_test, tf_plain),
    batch_size=512, shuffle=False, num_workers=2, pin_memory=True,
)
print(f'Train: {N_TRAIN}  Test: {N_TEST}  Classes: {CLASSES}')

100%|██████████| 170M/170M [00:04<00:00, 35.0MB/s]


Train: 50000  Test: 10000  Classes: ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']


In [4]:
import os
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        print(os.path.join(root, f))

/kaggle/input/datasets/dsvani/typiclust-simclr-cifar10/data/cifar-10-batches-py/data_batch_1
/kaggle/input/datasets/dsvani/typiclust-simclr-cifar10/data/cifar-10-batches-py/data_batch_2
/kaggle/input/datasets/dsvani/typiclust-simclr-cifar10/data/cifar-10-batches-py/batches.meta
/kaggle/input/datasets/dsvani/typiclust-simclr-cifar10/data/cifar-10-batches-py/test_batch
/kaggle/input/datasets/dsvani/typiclust-simclr-cifar10/data/cifar-10-batches-py/data_batch_3
/kaggle/input/datasets/dsvani/typiclust-simclr-cifar10/data/cifar-10-batches-py/data_batch_5
/kaggle/input/datasets/dsvani/typiclust-simclr-cifar10/data/cifar-10-batches-py/data_batch_4
/kaggle/input/datasets/dsvani/typiclust-simclr-cifar10/data/cifar-10-batches-py/readme.html
/kaggle/input/datasets/dsvani/typiclust-simclr-cifar10/data/cifar-10-python/cifar-10-batches-py/data_batch_1
/kaggle/input/datasets/dsvani/typiclust-simclr-cifar10/data/cifar-10-python/cifar-10-batches-py/data_batch_2
/kaggle/input/datasets/dsvani/typiclust-s

In [5]:
# ── Load pre-trained SimCLR checkpoint from uploaded dataset ──────────────────
_CACHE = '/kaggle/input/datasets/dsvani/typiclust-simclr-cifar10/typiclust_ckpts'
SIMCLR_CKPT = os.path.join(SAVE_DIR, 'simclr_model.pt')   # writable working dir
EMBED_PATH  = os.path.join(SAVE_DIR, 'embeddings.npy')
SIMCLR_LOG  = os.path.join(SAVE_DIR, 'simclr_loss_log.json')

import shutil
for _src_name, _dst in [
    ('simclr_model.pt',        SIMCLR_CKPT),
    ('embeddings.npy',         EMBED_PATH),
    ('simclr_loss_log.json',   SIMCLR_LOG),
]:
    _src = os.path.join(_CACHE, _src_name)
    if os.path.exists(_src) and not os.path.exists(_dst):
        shutil.copy2(_src, _dst)
        print(f'Copied {_src_name} -> {_dst}')
    elif os.path.exists(_dst):
        print(f'Already in working dir: {_src_name}')
    else:
        print(f'WARNING: not found in dataset: {_src}')

Copied simclr_model.pt -> /kaggle/working/typiclust_ckpts/simclr_model.pt
Copied embeddings.npy -> /kaggle/working/typiclust_ckpts/embeddings.npy
Copied simclr_loss_log.json -> /kaggle/working/typiclust_ckpts/simclr_loss_log.json


In [6]:
class SimCLRModel(nn.Module):
    """ResNet-18 backbone + two-layer MLP projector (Chen et al. 2020).
    For 32×32 CIFAR: replace 7×7 conv with 3×3 and remove maxpool.
    """
    def __init__(self, proj_dim: int = 128):
        super().__init__()
        self.backbone = models.resnet18(weights=None)
        # CIFAR adaptation (paper Appendix F.1)
        self.backbone.conv1   = nn.Conv2d(3, 64, 3, 1, 1, bias=False)
        self.backbone.maxpool = nn.Identity()
        feat_dim = self.backbone.fc.in_features  # 512
        self.backbone.fc = nn.Identity()
        # Two-layer MLP projector
        self.projector = nn.Sequential(
            nn.Linear(feat_dim, feat_dim),
            nn.ReLU(inplace=True),
            nn.Linear(feat_dim, proj_dim),
        )

    def forward(self, x, return_feat: bool = False):
        feat = self.backbone(x)          # (N, 512)
        if return_feat:
            return F.normalize(feat, dim=1)   # L2-normalised 512-d embedding
        return F.normalize(self.projector(feat), dim=1)  # 128-d projection


class NTXentLoss(nn.Module):
    """Normalised temperature-scaled cross-entropy (NT-Xent, Chen et al. 2020)."""
    def __init__(self, temperature: float = 0.5):
        super().__init__()
        self.T = temperature

    def forward(self, z1, z2):
        N = z1.size(0)
        z   = torch.cat([z1, z2], dim=0)                    # (2N, D)
        sim = torch.mm(z, z.t()) / self.T                   # (2N, 2N)
        sim.masked_fill_(
            torch.eye(2 * N, dtype=torch.bool, device=z.device), float('-inf'))
        pos = torch.cat([torch.arange(N, 2*N),
                          torch.arange(N)]).to(z.device)
        return F.cross_entropy(sim, pos)


class SimCLRDataset(Dataset):
    """here iam returtning the two augmented views of the same image."""
    def __init__(self, raw):
        self.raw = raw
    def __len__(self):
        return len(self.raw)
    def __getitem__(self, i):
        img, _ = self.raw[i]
        return tf_simclr(img), tf_simclr(img)

print('SimCLR model definition ready.')

SimCLR model definition ready.


In [7]:
_CACHE = '/kaggle/input/datasets/dsvani/typiclust-simclr-cifar10/typiclust_ckpts'
SIMCLR_CKPT  = os.path.join(_CACHE, 'simclr_model.pt')
EMBED_PATH   = os.path.join(_CACHE, 'embeddings.npy')
SIMCLR_LOG   = os.path.join(_CACHE, 'simclr_loss_log.json')

def train_simclr():
    if os.path.exists(SIMCLR_CKPT) and os.path.exists(EMBED_PATH):
        print('SimCLR checkpoint found – skipping training.')
        return

    epochs = CFG['simclr_epochs']
    print(f'Training SimCLR for {epochs} epochs on {DEVICE} ...')
    t0 = time.time()

    loader = DataLoader(
        SimCLRDataset(raw_train),
        batch_size=CFG['simclr_batch'], shuffle=True,
        num_workers=2, pin_memory=True, drop_last=True,
    )
    model     = SimCLRModel(CFG['simclr_proj_dim']).to(DEVICE)
    criterion = NTXentLoss(CFG['simclr_temperature'])
    optimizer = optim.SGD(
        model.parameters(),
        lr=CFG['simclr_lr'],
        momentum=CFG['simclr_momentum'],
        weight_decay=CFG['simclr_wd'],
    )
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    loss_log = []
    for ep in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        for v1, v2 in loader:
            v1, v2 = v1.to(DEVICE), v2.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(v1), model(v2))
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        scheduler.step()
        avg_loss = total_loss / len(loader)
        loss_log.append(avg_loss)
        if ep % 10 == 0 or ep == 1:
            print(f'  Ep {ep:3d}/{epochs}  loss={avg_loss:.4f}  '
                  f'lr={scheduler.get_last_lr()[0]:.4f}  '
                  f't={time.time()-t0:.0f}s')

    # Save model checkpoint
    torch.save({
        'model_state': model.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'epochs_trained': epochs,
        'cfg': {k: v for k, v in CFG.items() if 'simclr' in k},
    }, SIMCLR_CKPT)
    json.dump(loss_log, open(SIMCLR_LOG, 'w'))
    print(f'Checkpoint saved -> {SIMCLR_CKPT}')

    # Extract and save 512-d L2-normalised embeddings 
    print('Extracting embeddings for all 50 000 training imag')
    model.eval()
    embed_loader = DataLoader(
        PlainDataset(raw_train, tf_plain),
        batch_size=512, shuffle=False, num_workers=2, pin_memory=True,
    )
    feats = []
    with torch.no_grad():
        for x, _ in embed_loader:
            feats.append(model(x.to(DEVICE), return_feat=True).cpu().numpy())
    embs = np.concatenate(feats, axis=0)   # (50000, 512)
    np.save(EMBED_PATH, embs)
    print(f'Embeddings saved -> {EMBED_PATH}  shape={embs.shape}  '
          f'total_time={time.time()-t0:.0f}s')


train_simclr()

SimCLR checkpoint found – skipping training.


In [8]:
def load_simclr():
    assert os.path.exists(SIMCLR_CKPT), f'Missing: {SIMCLR_CKPT}'
    assert os.path.exists(EMBED_PATH),  f'Missing: {EMBED_PATH}'
    ckpt  = torch.load(SIMCLR_CKPT, map_location=DEVICE)
    model = SimCLRModel(CFG['simclr_proj_dim'])
    model.load_state_dict(ckpt['model_state'])
    model = model.to(DEVICE).eval()
    embs  = np.load(EMBED_PATH)
    print(f'SimCLR loaded  epochs={ckpt["epochs_trained"]}  '
          f'embedding shape={embs.shape}  device={DEVICE}')
    return model, embs


simclr_model, embeddings = load_simclr()

# Plot SimCLR training loss 
if os.path.exists(SIMCLR_LOG):
    loss_log = json.load(open(SIMCLR_LOG))
    fig, ax = plt.subplots(figsize=(7, 3))
    ax.plot(range(1, len(loss_log)+1), loss_log, color='steelblue', lw=1.5)
    ax.set_xlabel('Epoch'); ax.set_ylabel('NT-Xent Loss')
    ax.set_title('SimCLR Pre-training Loss (100 epochs, CIFAR-10)')
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    fig.savefig(os.path.join(PLOT_DIR, 'simclr_loss.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print('Plot saved: simclr_loss.png')

SimCLR loaded  epochs=100  embedding shape=(50000, 512)  device=cuda
Plot saved: simclr_loss.png


In [9]:
def compute_typicality(embs_subset: np.ndarray, k: int = 20) -> np.ndarray:
    """Eq. (4) from the paper: inverse mean k-NN distance."""
    k_eff = min(k + 1, len(embs_subset))
    nn = NearestNeighbors(n_neighbors=k_eff, metric='euclidean', algorithm='auto')
    nn.fit(embs_subset)
    dists, _ = nn.kneighbors(embs_subset)
    mean_dists = dists[:, 1:].mean(axis=1)
    mean_dists = np.clip(mean_dists, 1e-10, None)
    return 1.0 / mean_dists


def typiclust_rp_query(
    embeddings_all: np.ndarray,
    existing_labeled: list,
    budget: int,
    max_clusters: int = 500,
    k: int = 20,
) -> list:
    
    
    K = min(len(existing_labeled) + budget, max_clusters)
    K = max(K, budget) 
    
    if K <= 50:
        km = KMeans(n_clusters=K, random_state=SEED, n_init=10)
    else:
        km = MiniBatchKMeans(n_clusters=K, random_state=SEED,
                             batch_size=2048, n_init=5)
    cluster_ids = km.fit_predict(embeddings_all)   # shape (N,)

    labeled_set = set(existing_labeled)

    
    covered_clusters = set(cluster_ids[i] for i in existing_labeled)
    uncovered = [c for c in range(K) if c not in covered_clusters]

    
    cluster_sizes   = {c: int((cluster_ids == c).sum()) for c in uncovered}
    sorted_clusters = sorted(uncovered, key=lambda c: cluster_sizes[c], reverse=True)
    top_clusters    = sorted_clusters[:budget]

    queries = []
    for c in top_clusters:
        member_idx = np.where(cluster_ids == c)[0]
        member_idx = [i for i in member_idx if i not in labeled_set]
        if len(member_idx) < 2:
            if len(member_idx) == 1:
                queries.append(int(member_idx[0]))
                labeled_set.add(int(member_idx[0]))
            continue
        member_embs = embeddings_all[member_idx]
        typ_scores  = compute_typicality(member_embs, k=min(k, len(member_idx)-1))
        best_local  = int(np.argmax(typ_scores))
        queries.append(int(member_idx[best_local]))
        labeled_set.add(int(member_idx[best_local]))

    return queries


print('TPC_RP fixed created and is defined.')

TPC_RP fixed created and is defined.


In [10]:
TYPI_PATH = os.path.join(SAVE_DIR, 'typi_labeled.json')
RAND_PATH = os.path.join(SAVE_DIR, 'rand_labeled.json')

def build_query_lists():
    if os.path.exists(TYPI_PATH) and os.path.exists(RAND_PATH):
        typi = json.load(open(TYPI_PATH))
        rand = json.load(open(RAND_PATH))
        print(f'Query lists loaded  typi={len(typi)}  rand={len(rand)}')
        return typi, rand

    typi_labeled: list = []
    rand_labeled: list = []
    B    = CFG['al_budget']
    iters = CFG['al_iterations']
    print(f'Running TPC_RP active selection  (B={B}, iters={iters}) ...')

    for it in range(1, iters + 1):
        t0 = time.time()
        new_typi = typiclust_rp_query(
            embeddings,
            existing_labeled=typi_labeled,
            budget=B,
            max_clusters=CFG['max_clusters'],
            k=CFG['typi_k'],
        )
        typi_labeled.extend(new_typi)

         
        remaining = list(set(ALL_TRAIN_IDX) - set(rand_labeled))
        new_rand  = random.sample(remaining, B)
        rand_labeled.extend(new_rand)

        print(f'  Iter {it}: +{len(new_typi)} typi  +{len(new_rand)} rand  '
              f'(total typi={len(typi_labeled)})  {time.time()-t0:.1f}s')


    json.dump(typi_labeled, open(TYPI_PATH, 'w'), indent=2)
    json.dump(rand_labeled, open(RAND_PATH,  'w'), indent=2)
    torch.save({'typi_labeled': typi_labeled, 'rand_labeled': rand_labeled},
               os.path.join(SAVE_DIR, 'query_indices.pt'))
    print(f'Saved query lists -> {TYPI_PATH}')
    return typi_labeled, rand_labeled


typi_all, rand_all = build_query_lists()

# Class-balance 
def class_balance(indices):
    labels = [raw_train[i][1] for i in indices]
    counts = [labels.count(c) for c in range(N_CLS)]
    return counts

typi_balance = class_balance(typi_all)
rand_balance = class_balance(rand_all)
tv_typi = 0.5 * sum(abs(c/len(typi_all) - 1/N_CLS) for c in typi_balance)
tv_rand = 0.5 * sum(abs(c/len(rand_all) - 1/N_CLS) for c in rand_balance)
print(f'\nClass balance check (lower TV = more balanced):')
print(f'  TPC_RP TV distance = {tv_typi:.4f}')
print(f'  Random TV distance = {tv_rand:.4f}')
print(f'  TPC_RP per-class: {typi_balance}')
print(f'  Random  per-class: {rand_balance}')

Running TPC_RP active selection  (B=10, iters=5) ...
  Iter 1: +10 typi  +10 rand  (total typi=10)  21.8s
  Iter 2: +10 typi  +10 rand  (total typi=20)  32.4s
  Iter 3: +10 typi  +10 rand  (total typi=30)  42.4s
  Iter 4: +10 typi  +10 rand  (total typi=40)  52.2s
  Iter 5: +10 typi  +10 rand  (total typi=50)  62.8s
Saved query lists -> /kaggle/working/typiclust_ckpts/typi_labeled.json

Class balance check (lower TV = more balanced):
  TPC_RP TV distance = 0.0800
  Random TV distance = 0.1200
  TPC_RP per-class: [7, 5, 6, 3, 5, 6, 5, 5, 4, 4]
  Random  per-class: [2, 6, 6, 6, 6, 4, 5, 7, 5, 3]


In [11]:
TSNE_PLOT = os.path.join(PLOT_DIR, 'tsne_typiclust_selection.png')

def plot_tsne_selection():
    sample_idx = np.random.choice(N_TRAIN, 5000, replace=False)
    sample_embs = embeddings[sample_idx]
    selected_in_sample = [np.where(sample_idx == i)[0][0]
                          for i in typi_all if i in set(sample_idx)]

    print('Running t-SNE (this may take ~2 min) ...')
    t0 = time.time()
    tsne = TSNE(n_components=2, random_state=SEED, perplexity=30, n_iter=500)
    emb2d = tsne.fit_transform(sample_embs)
    labels_sample = [raw_train[i][1] for i in sample_idx]
    print(f't-SNE done in {time.time()-t0:.0f}s')

    cmap = plt.cm.get_cmap('tab10', 10)
    fig, ax = plt.subplots(figsize=(8, 6))
    for c in range(N_CLS):
        mask = np.array(labels_sample) == c
        ax.scatter(emb2d[mask, 0], emb2d[mask, 1],
                   c=[cmap(c)], s=2, alpha=0.4, label=CLASSES[c])

    if selected_in_sample:
        sel_pts = emb2d[selected_in_sample]
        ax.scatter(sel_pts[:, 0], sel_pts[:, 1],
                   c='black', s=80, marker='x', linewidths=1.5,
                   zorder=5, label='TPC_RP selected')

    ax.legend(markerscale=3, fontsize=7, ncol=2, loc='upper right')
    ax.set_title(f't-SNE of SimCLR embeddings (5k subset)\n'
                 f'× marks TPC_RP selected examples (n={len(selected_in_sample)})')
    ax.axis('off')
    fig.tight_layout()
    fig.savefig(TSNE_PLOT, dpi=150, bbox_inches='tight')
    plt.show()
    print(f't-SNE plot saved -> {TSNE_PLOT}')

plot_tsne_selection()

Running t-SNE (this may take ~2 min) ...
t-SNE done in 12s
t-SNE plot saved -> /kaggle/working/plots/tsne_typiclust_selection.png


In [12]:
RESULTS_A = os.path.join(SAVE_DIR, 'results_A.json')

def build_resnet18() -> nn.Module:
    net = models.resnet18(weights=None)
    net.conv1   = nn.Conv2d(3, 64, 3, 1, 1, bias=False)
    net.maxpool = nn.Identity()
    net.fc      = nn.Linear(net.fc.in_features, N_CLS)
    return net


class LabeledDataset(Dataset):
    def __init__(self, indices, transform):
        self.data = [(raw_train[i][0], raw_train[i][1]) for i in indices]
        self.tf   = transform
    def __len__(self):
        return len(self.data)
    def __getitem__(self, i):
        img, label = self.data[i]
        return self.tf(img), label


@torch.no_grad()
def evaluate(net: nn.Module) -> float:
    net.eval()
    correct = total = 0
    for x, y in test_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        correct += (net(x).argmax(1) == y).sum().item()
        total   += y.size(0)
    return round(100.0 * correct / total, 2)


def train_supervised(labeled_indices: list, tag: str = '') -> float:
    print(f'  [A-{tag}] n_labeled={len(labeled_indices)}')
    ds     = LabeledDataset(labeled_indices, tf_train)
    loader = DataLoader(ds, batch_size=min(CFG['sup_batch'], len(ds)),
                        shuffle=True, num_workers=2, pin_memory=True, drop_last=False)
   
    net   = build_resnet18().to(DEVICE)
    opt   = optim.SGD(net.parameters(), lr=CFG['sup_lr'],
                      momentum=CFG['sup_momentum'], nesterov=True, weight_decay=1e-4)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=CFG['sup_epochs'])

    for ep in range(CFG['sup_epochs']):
        net.train()
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad()
            F.cross_entropy(net(x), y).backward()
            opt.step()
        sched.step()

    acc = evaluate(net)
    print(f'  [A-{tag}] acc={acc:.2f}%')
    del net; torch.cuda.empty_cache()
    return acc


def run_framework_A():
    results = (json.load(open(RESULTS_A))
               if os.path.exists(RESULTS_A)
               else {'typiclust': [], 'random': [], 'n_labels': []})
    done = len(results['typiclust'])

    for it in range(1, CFG['al_iterations'] + 1):
        if it <= done:
            print(f'[A] Iteration {it} already done – skipping.')
            continue
        n = it * CFG['al_budget']
        print(f'\n[Framework A] Iteration {it}  ({n} labeled examples)')
        acc_t = train_supervised(typi_all[:n], tag='TypiClust')
        acc_r = train_supervised(rand_all[:n], tag='Random')
        results['typiclust'].append(acc_t)
        results['random'].append(acc_r)
        results['n_labels'].append(n)
        json.dump(results, open(RESULTS_A, 'w'), indent=2)
        torch.save(results, os.path.join(SAVE_DIR, 'results_A.pt'))
        print(f'  Gain = {acc_t - acc_r:+.2f}%  [saved]')

    print(f'\n[Framework A] Complete. Results: {RESULTS_A}')
    return results


results_A = run_framework_A()


[Framework A] Iteration 1  (10 labeled examples)
  [A-TypiClust] n_labeled=10
  [A-TypiClust] acc=16.85%
  [A-Random] n_labeled=10
  [A-Random] acc=15.83%
  Gain = +1.02%  [saved]

[Framework A] Iteration 2  (20 labeled examples)
  [A-TypiClust] n_labeled=20
  [A-TypiClust] acc=21.46%
  [A-Random] n_labeled=20
  [A-Random] acc=16.93%
  Gain = +4.53%  [saved]

[Framework A] Iteration 3  (30 labeled examples)
  [A-TypiClust] n_labeled=30
  [A-TypiClust] acc=23.15%
  [A-Random] n_labeled=30
  [A-Random] acc=20.21%
  Gain = +2.94%  [saved]

[Framework A] Iteration 4  (40 labeled examples)
  [A-TypiClust] n_labeled=40
  [A-TypiClust] acc=23.56%
  [A-Random] n_labeled=40
  [A-Random] acc=20.37%
  Gain = +3.19%  [saved]

[Framework A] Iteration 5  (50 labeled examples)
  [A-TypiClust] n_labeled=50
  [A-TypiClust] acc=22.69%
  [A-Random] n_labeled=50
  [A-Random] acc=20.11%
  Gain = +2.58%  [saved]

[Framework A] Complete. Results: /kaggle/working/typiclust_ckpts/results_A.json


In [13]:
RESULTS_B = os.path.join(SAVE_DIR, 'results_B.json')

def get_test_embeddings() -> torch.Tensor:
    simclr_model.eval()
    feats = []
    with torch.no_grad():
        for x, _ in test_loader:
            feats.append(simclr_model(x.to(DEVICE), return_feat=True).cpu())
    return torch.cat(feats, dim=0)



_test_embs = get_test_embeddings()
_test_lbls = torch.tensor([raw_test[i][1] for i in range(N_TEST)])
print(f'Test embeddings: {_test_embs.shape}')


def train_linear_probe(labeled_indices: list, tag: str = '') -> float:
    print(f'  [B-{tag}] n_labeled={len(labeled_indices)}')
    X_tr = torch.tensor(embeddings[labeled_indices], dtype=torch.float32)
    y_tr = torch.tensor([raw_train[i][1] for i in labeled_indices], dtype=torch.long)

    feat_dim = embeddings.shape[1]  # 512
    lp  = nn.Linear(feat_dim, N_CLS).to(DEVICE)
    opt = optim.SGD(lp.parameters(), lr=CFG['lp_lr'], momentum=0.9, weight_decay=0.0)
    sch = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=CFG['lp_epochs'])

    for _ in range(CFG['lp_epochs']):
        lp.train()
        perm = torch.randperm(len(X_tr))
        for i in range(0, max(len(X_tr), 1), CFG['lp_batch']):
            xb = X_tr[perm[i: i + CFG['lp_batch']]].to(DEVICE)
            yb = y_tr[perm[i: i + CFG['lp_batch']]].to(DEVICE)
            opt.zero_grad()
            F.cross_entropy(lp(xb), yb).backward()
            opt.step()
        sch.step()

    lp.eval()
    correct = total = 0
    with torch.no_grad():
        for i in range(0, len(_test_embs), 512):
            xb = _test_embs[i: i + 512].to(DEVICE)
            yb = _test_lbls[i: i + 512].to(DEVICE)
            correct += (lp(xb).argmax(1) == yb).sum().item()
            total   += yb.size(0)
    acc = round(100.0 * correct / total, 2)
    print(f'  [B-{tag}] acc={acc:.2f}%')
    del lp; torch.cuda.empty_cache()
    return acc


def run_framework_B():
    results = (json.load(open(RESULTS_B))
               if os.path.exists(RESULTS_B)
               else {'typiclust': [], 'random': [], 'n_labels': []})
    done = len(results['typiclust'])

    for it in range(1, CFG['al_iterations'] + 1):
        if it <= done:
            print(f'[B] Iteration {it} already done – skipping.')
            continue
        n = it * CFG['al_budget']
        print(f'\n[Framework B] Iteration {it}  ({n} labeled examples)')
        acc_t = train_linear_probe(typi_all[:n], tag='TypiClust')
        acc_r = train_linear_probe(rand_all[:n], tag='Random')
        results['typiclust'].append(acc_t)
        results['random'].append(acc_r)
        results['n_labels'].append(n)
        json.dump(results, open(RESULTS_B, 'w'), indent=2)
        torch.save(results, os.path.join(SAVE_DIR, 'results_B.pt'))
        print(f'  Gain = {acc_t - acc_r:+.2f}%  [saved]')

    print(f'\n[Framework B] Complete. Results: {RESULTS_B}')
    return results


results_B = run_framework_B()

Test embeddings: torch.Size([10000, 512])

[Framework B] Iteration 1  (10 labeled examples)
  [B-TypiClust] n_labeled=10
  [B-TypiClust] acc=61.57%
  [B-Random] n_labeled=10
  [B-Random] acc=43.72%
  Gain = +17.85%  [saved]

[Framework B] Iteration 2  (20 labeled examples)
  [B-TypiClust] n_labeled=20
  [B-TypiClust] acc=70.47%
  [B-Random] n_labeled=20
  [B-Random] acc=56.99%
  Gain = +13.48%  [saved]

[Framework B] Iteration 3  (30 labeled examples)
  [B-TypiClust] n_labeled=30
  [B-TypiClust] acc=72.36%
  [B-Random] n_labeled=30
  [B-Random] acc=63.30%
  Gain = +9.06%  [saved]

[Framework B] Iteration 4  (40 labeled examples)
  [B-TypiClust] n_labeled=40
  [B-TypiClust] acc=73.59%
  [B-Random] n_labeled=40
  [B-Random] acc=63.58%
  Gain = +10.01%  [saved]

[Framework B] Iteration 5  (50 labeled examples)
  [B-TypiClust] n_labeled=50
  [B-TypiClust] acc=74.70%
  [B-Random] n_labeled=50
  [B-Random] acc=63.65%
  Gain = +11.05%  [saved]

[Framework B] Complete. Results: /kaggle/working

In [14]:
RESULTS_C = os.path.join(SAVE_DIR, 'results_C.json')

class UnlabeledDataset(Dataset):
    def __init__(self, indices):
        self.images = [raw_train[i][0] for i in indices]
    def __len__(self):
        return len(self.images)
    def __getitem__(self, i):
        return tf_weak(self.images[i]), tf_strong(self.images[i])


def train_semi_supervised(labeled_indices: list, tag: str = '') -> float:
    print(f'  [C-{tag}] n_labeled={len(labeled_indices)}')
    unlabeled_idx = [i for i in ALL_TRAIN_IDX if i not in set(labeled_indices)]

    lab_ds  = LabeledDataset(labeled_indices, tf_weak)
    unl_ds  = UnlabeledDataset(unlabeled_idx)
    lab_loader = DataLoader(lab_ds,
                            batch_size=min(CFG['ssl_batch_l'], len(lab_ds)),
                            shuffle=True, num_workers=2, drop_last=False)
    unl_loader = DataLoader(unl_ds, batch_size=CFG['ssl_batch_u'],
                            shuffle=True, num_workers=2, drop_last=True)

    net = build_resnet18().to(DEVICE)
    opt = optim.SGD(net.parameters(), lr=CFG['ssl_lr'],
                    momentum=0.9, weight_decay=CFG['ssl_wd'], nesterov=True)

    lab_iter = itertools.cycle(lab_loader)
    unl_iter = itertools.cycle(unl_loader)
    net.train()

    log_steps = []
    total_iters = CFG['ssl_iters']
    for step in range(1, total_iters + 1):
        x_l, y_l     = next(lab_iter)
        x_uw, x_us   = next(unl_iter)
        x_l, y_l     = x_l.to(DEVICE), y_l.to(DEVICE)
        x_uw, x_us   = x_uw.to(DEVICE), x_us.to(DEVICE)

        loss_sup = F.cross_entropy(net(x_l), y_l)

        with torch.no_grad():
            probs       = F.softmax(net(x_uw), dim=1)
            max_p, pseudo_y = probs.max(dim=1)
            mask        = max_p.ge(CFG['ssl_threshold']).float()

        loss_unl = (F.cross_entropy(net(x_us), pseudo_y,
                                    reduction='none') * mask).mean()
        opt.zero_grad()
        (loss_sup + loss_unl).backward()
        opt.step()

        if step % 5000 == 0 or step == total_iters:
            acc = evaluate(net); net.train()
            log_steps.append((step, acc))
            print(f'    step {step}/{total_iters}  acc={acc:.1f}%')

    acc = evaluate(net)
    print(f'  [C-{tag}] final acc={acc:.2f}%')
    del net; torch.cuda.empty_cache()
    return acc


def run_framework_C():
    results = (json.load(open(RESULTS_C))
               if os.path.exists(RESULTS_C)
               else {'typiclust': [], 'random': [], 'n_labels': []})
    done = len(results['typiclust'])

    for it in range(1, CFG['al_iterations'] + 1):
        if it <= done:
            print(f'[C] Iteration {it} already done – skipping.')
            continue
        n = it * CFG['al_budget']
        print(f'\n[Framework C] Iteration {it}  ({n} labeled examples)')
        acc_t = train_semi_supervised(typi_all[:n], tag='TypiClust')
        acc_r = train_semi_supervised(rand_all[:n], tag='Random')
        results['typiclust'].append(acc_t)
        results['random'].append(acc_r)
        results['n_labels'].append(n)
        json.dump(results, open(RESULTS_C, 'w'), indent=2)
        torch.save(results, os.path.join(SAVE_DIR, 'results_C.pt'))
        print(f'  Gain = {acc_t - acc_r:+.2f}%  [saved]')

    print(f'\n[Framework C] Complete. Results: {RESULTS_C}')
    return results


results_C = run_framework_C()


[Framework C] Iteration 1  (10 labeled examples)
  [C-TypiClust] n_labeled=10
    step 5000/30000  acc=10.0%
    step 10000/30000  acc=13.3%
    step 15000/30000  acc=16.4%
    step 20000/30000  acc=10.0%
    step 25000/30000  acc=10.0%
    step 30000/30000  acc=10.0%
  [C-TypiClust] final acc=10.00%
  [C-Random] n_labeled=10
    step 5000/30000  acc=10.0%
    step 10000/30000  acc=10.0%
    step 15000/30000  acc=10.0%
    step 20000/30000  acc=10.0%
    step 25000/30000  acc=10.0%
    step 30000/30000  acc=10.0%
  [C-Random] final acc=10.01%
  Gain = -0.01%  [saved]

[Framework C] Iteration 2  (20 labeled examples)
  [C-TypiClust] n_labeled=20
    step 5000/30000  acc=10.0%
    step 10000/30000  acc=10.0%
    step 15000/30000  acc=10.0%
    step 20000/30000  acc=15.2%
    step 25000/30000  acc=10.0%
    step 30000/30000  acc=10.0%
  [C-TypiClust] final acc=10.00%
  [C-Random] n_labeled=20
    step 5000/30000  acc=10.0%
    step 10000/30000  acc=10.0%
    step 15000/30000  acc=10.0%
 

In [15]:
def load_r(path):
    if not os.path.exists(path):
        return None
    return json.load(open(path))

rA = load_r(RESULTS_A)
rB = load_r(RESULTS_B)
rC = load_r(RESULTS_C)

colors = {'typiclust': '#2196F3', 'random': '#F44336'}
fw_labels = {
    'A': 'Fully Supervised',
    'B': 'Linear Probe (SSL Emb.)',
    'C': 'Semi-Supervised',
}

# Plot 1: Accuracy vs Budget
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=False)
for ax, (key, res, fw) in zip(axes, [('A', rA, 'A'), ('B', rB, 'B'), ('C', rC, 'C')]):
    if res is None:
        ax.text(0.5, 0.5, 'Not run yet', ha='center', va='center',
                transform=ax.transAxes)
        ax.set_title(fw_labels[fw]); continue
    ns = res['n_labels']
    ax.plot(ns, res['typiclust'], 'o-', color=colors['typiclust'],
            lw=2, ms=7, label='TPC_RP (Ours)')
    ax.plot(ns, res['random'],    's--', color=colors['random'],
            lw=2, ms=7, label='Random')
    ax.set_xlabel('Cumulative Budget (# labels)')
    ax.set_ylabel('Test Accuracy (%)')
    ax.set_title(f'Framework {fw_labels[fw]}')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.set_xticks(ns)

fig.suptitle('TPC_RP vs Random – CIFAR-10 (Low Budget Regime)', fontsize=13, fontweight='bold')
fig.tight_layout()
fig.savefig(os.path.join(PLOT_DIR, 'plot1_accuracy_vs_budget.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Plot 1 saved.')

# Plot 2: Accuracy Gain (phase transition visualisation)
fig, ax = plt.subplots(figsize=(7, 4))
style = [('A', '-',  'o'), ('B', '--', 's'), ('C', ':',  '^')]
palette = ['#1565C0', '#2E7D32', '#AD1457']
for (fw, ls, mk), col, res in zip(style, palette, [rA, rB, rC]):
    if res is None: continue
    gains = [t - r for t, r in zip(res['typiclust'], res['random'])]
    ax.plot(res['n_labels'], gains, linestyle=ls, marker=mk, color=col,
            lw=2, ms=7, label=f'Fw {fw}: {fw_labels[fw]}')
ax.axhline(0, color='black', lw=1, linestyle='-')
ax.fill_between(ax.get_xlim(), 0, ax.get_ylim()[1] if ax.get_ylim()[1] > 0 else 5,
                alpha=0.05, color='green', label='TPC_RP > Random')
ax.set_xlabel('Cumulative Budget (# labels)')
ax.set_ylabel('Accuracy Gain (TPC_RP − Random) %')
ax.set_title('Accuracy Gain of TPC_RP over Random Baseline')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(os.path.join(PLOT_DIR, 'plot2_accuracy_gain.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Plot 2 saved.')

# Plot 3: Class Balance
x = np.arange(N_CLS)
w = 0.35
fig, ax = plt.subplots(figsize=(9, 3.5))
ax.bar(x - w/2, typi_balance, w, label='TPC_RP', color=colors['typiclust'], alpha=0.85)
ax.bar(x + w/2, rand_balance, w, label='Random', color=colors['random'],    alpha=0.85)
ax.axhline(len(typi_all) / N_CLS, color='gray', lw=1.5, linestyle='--', label='Ideal balance')
ax.set_xticks(x); ax.set_xticklabels(CLASSES, rotation=30, ha='right', fontsize=9)
ax.set_ylabel('# Selected Examples')
ax.set_title(f'Class Distribution of Selected Labels\n'
             f'TV-dist: TPC_RP={tv_typi:.3f}  Random={tv_rand:.3f} (lower = more balanced)')
ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(PLOT_DIR, 'plot3_class_balance.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Plot 3 saved.')

Plot 1 saved.
Plot 2 saved.
Plot 3 saved.


In [16]:
print('=' * 65)
print('  STATISTICAL ANALYSIS  –  TPC_RP vs Random')
print('=' * 65)

def bootstrap_ci(gains, n_boot=5000, ci=0.95):
    gains = np.array(gains)
    boot_means = [np.mean(np.random.choice(gains, size=len(gains), replace=True))
                  for _ in range(n_boot)]
    lo, hi = np.percentile(boot_means, [(1-ci)/2*100, (1+ci)/2*100])
    return np.mean(gains), lo, hi

def cohens_d(a, b):
    a, b = np.array(a), np.array(b)
    pooled_std = np.sqrt((np.std(a, ddof=1)**2 + np.std(b, ddof=1)**2) / 2)
    return (np.mean(a) - np.mean(b)) / (pooled_std + 1e-10)

for fw_name, res in [('A – Fully Supervised', rA),
                     ('B – Linear Probe',      rB),
                     ('C – Semi-Supervised',   rC)]:
    if res is None or len(res['typiclust']) < 2:
        print(f'\n[{fw_name}]  insufficient data')
        continue

    t_accs = np.array(res['typiclust'])
    r_accs = np.array(res['random'])
    gains  = t_accs - r_accs

    mean_g, lo, hi = bootstrap_ci(gains)
    d              = cohens_d(t_accs, r_accs)


    if len(gains) >= 3:
        try:
            stat, p_val = stats.wilcoxon(gains, alternative='greater')
        except ValueError:
            stat, p_val = float('nan'), float('nan')
    else:
        stat, p_val = float('nan'), float('nan')

    print(f'\n[{fw_name}]')
    print(f'  TPC_RP accs : {list(np.round(t_accs, 2))}')
    print(f'  Random  accs: {list(np.round(r_accs, 2))}')
    print(f'  Gains       : {list(np.round(gains, 2))}')
    print(f'  Mean gain   : {mean_g:+.2f}%  '
          f'95% CI [{lo:+.2f}%, {hi:+.2f}%]')
    print(f'  Wilcoxon    : stat={stat:.2f}  p={p_val:.4f}  '
          f'(significant at 0.05: {"YES" if p_val < 0.05 else "NO (low n)"} )')
    print(f'  Cohen\'s d   : {d:.3f}  '
          f'({"small" if abs(d)<0.5 else "medium" if abs(d)<0.8 else "large"})')

print('\n' + '=' * 65)
print('Note: n=5 iterations. With more repetitions (paper uses 10)')
print('statistical tests would be more powerful.')
print('=' * 65)

  STATISTICAL ANALYSIS  –  TPC_RP vs Random

[A – Fully Supervised]
  TPC_RP accs : [np.float64(16.85), np.float64(21.46), np.float64(23.15), np.float64(23.56), np.float64(22.69)]
  Random  accs: [np.float64(15.83), np.float64(16.93), np.float64(20.21), np.float64(20.37), np.float64(20.11)]
  Gains       : [np.float64(1.02), np.float64(4.53), np.float64(2.94), np.float64(3.19), np.float64(2.58)]
  Mean gain   : +2.85%  95% CI [+1.77%, +3.82%]
  Wilcoxon    : stat=15.00  p=0.0312  (significant at 0.05: YES )
  Cohen's d   : 1.159  (large)

[B – Linear Probe]
  TPC_RP accs : [np.float64(61.57), np.float64(70.47), np.float64(72.36), np.float64(73.59), np.float64(74.7)]
  Random  accs: [np.float64(43.72), np.float64(56.99), np.float64(63.3), np.float64(63.58), np.float64(63.65)]
  Gains       : [np.float64(17.85), np.float64(13.48), np.float64(9.06), np.float64(10.01), np.float64(11.05)]
  Mean gain   : +12.29%  95% CI [+9.84%, +15.23%]
  Wilcoxon    : stat=15.00  p=0.0312  (significant at

In [17]:
PAPER_RESULTS = {
    # From Fig. 4 
    'A_typiclust': [27.5, 30.2, 33.1, 36.4, 38.8],
    'A_random'   : [20.1, 22.0, 24.5, 26.3, 28.1],
    # From Figure 5
    'B_typiclust': [55.0, 60.2, 65.1, 68.3, 71.0],
    'B_random'   : [40.0, 45.0, 50.2, 53.1, 56.0],
    # From Fig 6a 
    'C_typiclust': [93.2],
    'C_random'   : [53.8],
}

B  = CFG['al_budget']
ns = list(range(B, B * CFG['al_iterations'] + 1, B))

print('=' * 75)
print('  RESULTS TABLE — TPC_RP vs RANDOM BASELINE on CIFAR-10')
print('  (Paper uses 500 SimCLR epochs / 200 sup epochs; ours: 100 / 70)')
print('=' * 75)

for fw_name, res, p_t, p_r in [
    ('A – Fully Supervised',  rA, 'A_typiclust', 'A_random'),
    ('B – Linear Probe (SSL)', rB, 'B_typiclust', 'B_random'),
    ('C – Semi-Supervised',    rC, 'C_typiclust', 'C_random'),
]:
    print(f'\n  Framework {fw_name}')
    print(f'  {"-"*70}')
    print(f'  {"Labels":>7}  '
          f'{"Ours-TPC_RP":>12}  {"Ours-Rand":>10}  '
          f'{"Gain":>6}  '
          f'{"Paper-TPC_RP":>13}  {"Paper-Rand":>10}')
    print(f'  {"-"*70}')

    if res is None:
        print('  (not run)')
        continue

    for i, n in enumerate(res['n_labels']):
        ot = res['typiclust'][i]
        or_ = res['random'][i]
        gain = ot - or_
        pt = PAPER_RESULTS[p_t][i] if i < len(PAPER_RESULTS[p_t]) else '-'
        pr = PAPER_RESULTS[p_r][i] if i < len(PAPER_RESULTS[p_r]) else '-'
        pt_s  = f'{pt:.1f}%' if isinstance(pt, float) else pt
        pr_s  = f'{pr:.1f}%' if isinstance(pr, float) else pr
        print(f'  {n:>7}  '
              f'{ot:>10.2f}%  {or_:>8.2f}%  '
              f'{gain:>+5.2f}%  '
              f'{pt_s:>13}  {pr_s:>10}')

print('\n' + '=' * 75)

  RESULTS TABLE — TPC_RP vs RANDOM BASELINE on CIFAR-10
  (Paper uses 500 SimCLR epochs / 200 sup epochs; ours: 100 / 70)

  Framework A – Fully Supervised
  ----------------------------------------------------------------------
   Labels   Ours-TPC_RP   Ours-Rand    Gain   Paper-TPC_RP  Paper-Rand
  ----------------------------------------------------------------------
       10       16.85%     15.83%  +1.02%          27.5%       20.1%
       20       21.46%     16.93%  +4.53%          30.2%       22.0%
       30       23.15%     20.21%  +2.94%          33.1%       24.5%
       40       23.56%     20.37%  +3.19%          36.4%       26.3%
       50       22.69%     20.11%  +2.58%          38.8%       28.1%

  Framework B – Linear Probe (SSL)
  ----------------------------------------------------------------------
   Labels   Ours-TPC_RP   Ours-Rand    Gain   Paper-TPC_RP  Paper-Rand
  ----------------------------------------------------------------------
       10       61.57%     43

In [18]:
def print_summary():
    B = CFG['al_budget']
    fw_info = [
        ('A – Fully Supervised',           RESULTS_A),
        ('B – Linear Probe (SSL Emb.)',     RESULTS_B),
        ('C – Semi-Supervised',             RESULTS_C),
    ]
    print('\n' + '=' * 65)
    print('  TYPICLUST TPC_RP vs RANDOM  –  CIFAR-10 RESULTS SUMMARY')
    print('=' * 65)
    for fw_name, path in fw_info:
        if not os.path.exists(path):
            print(f'\n[{fw_name}]  not yet run')
            continue
        res = json.load(open(path))
        print(f'\n[{fw_name}]')
        print(f'  {"Labels":>8}  {"TPC_RP":>9}  {"Random":>8}  {"Gain":>8}')
        print(f'  {"-" * 42}')
        for i, (t, r) in enumerate(zip(res['typiclust'], res['random'])):
            n = res['n_labels'][i]
            print(f'  {n:>8}  {t:>8.2f}%  {r:>7.2f}%  {t-r:>+7.2f}%')

    print('\n' + '=' * 65)
    print('  Checkpoints & outputs in:', SAVE_DIR)
    for fname in sorted(os.listdir(SAVE_DIR)):
        fpath = os.path.join(SAVE_DIR, fname)
        size  = os.path.getsize(fpath) / 1e6
        print(f'    {fname:<45} {size:.1f} MB')
    print('\n  Plots in:', PLOT_DIR)
    for fname in sorted(os.listdir(PLOT_DIR)):
        print(f'    {fname}')
    print('=' * 65)

print_summary()


  TYPICLUST TPC_RP vs RANDOM  –  CIFAR-10 RESULTS SUMMARY

[A – Fully Supervised]
    Labels     TPC_RP    Random      Gain
  ------------------------------------------
        10     16.85%    15.83%    +1.02%
        20     21.46%    16.93%    +4.53%
        30     23.15%    20.21%    +2.94%
        40     23.56%    20.37%    +3.19%
        50     22.69%    20.11%    +2.58%

[B – Linear Probe (SSL Emb.)]
    Labels     TPC_RP    Random      Gain
  ------------------------------------------
        10     61.57%    43.72%   +17.85%
        20     70.47%    56.99%   +13.48%
        30     72.36%    63.30%    +9.06%
        40     73.59%    63.58%   +10.01%
        50     74.70%    63.65%   +11.05%

[C – Semi-Supervised]
    Labels     TPC_RP    Random      Gain
  ------------------------------------------
        10     10.00%    10.01%    -0.01%
        20     10.00%    10.00%    +0.00%
        30     13.90%    10.00%    +3.90%
        40     18.09%    10.00%    +8.09%
        50    